# Site-Specific DUIDD Covariance Estimation Tutorial

This notebook covers the classical IDD and LMMSE channel-estimation experiments used to evaluate site-specific parameter adaptation.

The command cells below describe operations that can be computationally expensive. They are intentionally not executed automatically, and only the commands themselves are printed. Run these commands in a shell rather than from the notebook. Avoid other GPU-intensive workloads while the scripts are running, as GPU load can affect the results.


## Contents

- [1. Prerequisites and repository paths](#prerequisites)
- [2. Evaluate classical IDD schedules](#idd-evaluation)
- [3. Estimate covariance matrices](#covariance-estimation)
- [4. Evaluate LMMSE channel estimation](#lmmse-evaluation)
- [5. Examine the results](#results)
- [References](#references)


<a id="prerequisites"></a>

## 1. Prerequisites and Repository Paths

The required packages are listed in [`requirements.txt`](../requirements.txt). Check [`README.md`](../README.md) for the complete information.

Real-data steps require a running ClickHouse-backed Aerial Data Lake, databases containing the required `fapi` and `fh` tables, and NVIDIA pyAerial. The databases are not included in the repository; detailed information and download instructions are provided in [`datasets.md`](../../datasets.md).


In [2]:
from pathlib import Path
import os

candidates = (Path.cwd(), *Path.cwd().parents)
WORKSPACE_ROOT = next((path.resolve() for path in candidates if path.name == 'rx_training' and (path / 'nrx_duidd' / 'config').is_dir() and (path / 'nrx_duidd' / 'scripts').is_dir()), None)
if WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate the rx_training workspace root.')

os.chdir(WORKSPACE_ROOT)
REPO = Path('nrx_duidd')
DISPLAY_ROOT = Path('/rx_training')

print(f'Workspace root: {DISPLAY_ROOT}')
print(f'Repository: {DISPLAY_ROOT / REPO}')


Workspace root: /rx_training
Repository: /rx_training/nrx_duidd


In [7]:
from nrx_duidd.notebooks.notebook_helpers import NotebookHelpers

notebook = NotebookHelpers(WORKSPACE_ROOT, DISPLAY_ROOT)

idd_config = 'duidd_aerial_6_6.cfg'
synthetic_cov_config = 'duidd_12_lmmse_idd_umi.cfg'
lmmse_configs = (
    'duidd_12_lmmse_idd_umi.cfg',
    'duidd_12_lmmse_idd_hls.cfg',
)



<a id="idd-evaluation"></a>

### 2. Evaluate Classical IDD Performance

Use `--idd-only` to evaluate untuned classical IDD with the default untrained parameters. The configurations use schedules `[12]`, `[6, 6]`, `[4, 4, 4]`, and `[3, 3, 3, 3]`, corresponding to one through four MMSE-PIC detection stages. Each schedule keeps the total number of flooding min-sum LDPC message-passing iterations fixed at 12.

The command below shows the dual-layer `[6, 6]` evaluation. Change `idd_config` to one of the other configurations listed in the [DUIDD README](../README.md) to evaluate another schedule.


In [4]:
idd_command = f'''    cd {notebook.display_path(REPO / 'scripts')}
python eval_duidd_from_datalake.py --config-name {idd_config} --idd-only --db 8 --limit 100 --timestamps idd.pkl
'''.strip()

notebook.show_terminal(idd_command)
# os.system(idd_command)


```bash
$ cd /rx_training/nrx_duidd/scripts
$ python eval_duidd_from_datalake.py --config-name duidd_aerial_6_6.cfg --idd-only --db 8 --limit 100 --timestamps idd.pkl
```

<a id="covariance-estimation"></a>

### 3. Estimate Covariance Matrices

LMMSE channel estimation requires spatial, frequency, and temporal covariance matrices. [`compute_cov_mat.py`](../scripts/compute_cov_mat.py) estimates them from synthetic 3GPP UMi channel realizations.

The site-specific estimators form direct LS channel estimates at the actual DMRS positions without interpolation. They then average the measured correlations across three dimensions:

- [`estimate_pusch_spatial_covariance_from_datalake.py`](../cov_est/estimate_pusch_spatial_covariance_from_datalake.py) captures the correlation across the four receive antennas.
- [`estimate_pusch_frequency_covariance_from_datalake.py`](../cov_est/estimate_pusch_frequency_covariance_from_datalake.py) estimates correlations between pilot subcarriers using lag estimation and constructs a Toeplitz frequency covariance matrix.
- [`estimate_pusch_time_covariance_from_datalake.py`](../cov_est/estimate_pusch_time_covariance_from_datalake.py) estimates correlations across the three DMRS symbols and expands them across the OFDM resource grid using linear interpolation.

For each estimate, the channel-estimation error contribution is obtained from the noise-only fourteenth OFDM symbol and removed from the measured LS statistics. The commands below combine the Jun. 2026 Small Laboratory single- and dual-layer records selected by `cov_1ULL.pkl` and `cov_2ULL.pkl`.

The frequency estimator additionally calculates naive estimates obtained from linear interpolation; however, the experiments use the lag/Toeplitz results, which must be copied or linked to the label-matched `<label>_freq_cov_mat.npy` filename expected by the LMMSE configurations. The spatial and temporal scripts already write the expected suffixes. All schedule variants use the same three covariance estimates.

All estimated covariance matrices are made positive semidefinite (PSD) by projecting the negative eigenvalues to zero. The spatial and temporal estimators apply this projection by default. Pass `--project-lag-psd` to apply it to the lag/Toeplitz frequency estimate.

In [ ]:
synthetic_cov_command = f'''    cd {notebook.display_path(REPO / 'scripts')}
python compute_cov_mat.py -config_name {synthetic_cov_config}
'''.strip()

notebook.show_terminal(
    synthetic_cov_command,
    [('Covariance prefix', notebook.display_path(REPO / 'weights' / 'duidd_12_lmmse_idd_umi'))],
)
# os.system(synthetic_cov_command)


```bash
$ cd /rx_training/nrx_duidd/scripts
$ python compute_cov_mat.py -config_name duidd_12_lmmse_idd_umi.cfg
```

```text
Covariance prefix: /rx_training/nrx_duidd/weights/duidd_12_lmmse_idd_umi
```

In [6]:
site_specific_prefix = REPO / 'weights' / 'duidd_12_lmmse_idd_hls'
covariance_options = '--single-db 9 --dual-db 8 --single-timestamps ../eval_timestamps/cov_1ULL.pkl --dual-timestamps ../eval_timestamps/cov_2ULL.pkl --output-prefix ../weights/duidd_12_lmmse_idd_hls'
covariance_scripts = (
    'estimate_pusch_frequency_covariance_from_datalake.py',
    'estimate_pusch_time_covariance_from_datalake.py',
    'estimate_pusch_spatial_covariance_from_datalake.py',
)

for script in covariance_scripts:
    psd_option = ' --project-lag-psd' if script == 'estimate_pusch_frequency_covariance_from_datalake.py' else ''
    command = f'''        
cd {notebook.display_path(REPO / 'cov_est')}
python {script} {covariance_options}{psd_option}
    '''.strip()
    notebook.show_terminal(command)


```bash
$ cd /rx_training/nrx_duidd/cov_est
$ python estimate_pusch_frequency_covariance_from_datalake.py --single-db 9 --dual-db 8 --single-timestamps ../eval_timestamps/cov_1ULL.pkl --dual-timestamps ../eval_timestamps/cov_2ULL.pkl --output-prefix ../weights/duidd_12_lmmse_idd_hls --project-lag-psd
```

```bash
$ cd /rx_training/nrx_duidd/cov_est
$ python estimate_pusch_time_covariance_from_datalake.py --single-db 9 --dual-db 8 --single-timestamps ../eval_timestamps/cov_1ULL.pkl --dual-timestamps ../eval_timestamps/cov_2ULL.pkl --output-prefix ../weights/duidd_12_lmmse_idd_hls
```

```bash
$ cd /rx_training/nrx_duidd/cov_est
$ python estimate_pusch_spatial_covariance_from_datalake.py --single-db 9 --dual-db 8 --single-timestamps ../eval_timestamps/cov_1ULL.pkl --dual-timestamps ../eval_timestamps/cov_2ULL.pkl --output-prefix ../weights/duidd_12_lmmse_idd_hls
```

The provided [`plot_cov_mats.py`](../scripts/plot_cov_mats.py) script can be used to plot the covariance matrices for a visual inspection.

In [42]:
plot_command = f'''    cd {notebook.display_path(REPO / 'cov_est')}
python plot_cov_mats.py --prefix ../weights/duidd_12_lmmse_idd_hls --out cov_mat_heatmaps.png
'''.strip()
notebook.show_terminal(plot_command)

```bash
$ cd /rx_training/nrx_duidd/cov_est
$ python plot_cov_mats.py --prefix ../weights/duidd_12_lmmse_idd_hls --out cov_mat_heatmaps.png
```

The resulting heatmaps show the magnitude, logarithmic magnitude, and phase of the spatial, temporal, and frequency covariance matrices. They provide a visual check of the matrix dimensions and correlation structure before evaluation.

<p align="left">
  <img src="../../fig/duidd/freq_cov_mat_magnitude.png" alt="Site-specific spatial, temporal, and frequency covariance-matrix heatmaps" width="720">
</p>


<a id="lmmse-evaluation"></a>

### 4. Evaluate LMMSE Channel Estimation

Select a configuration whose filename contains `umi` to use the synthetic covariance estimates or `hls` to use the site-specific estimates. These configurations evaluate classical IDD and must be run with `--idd-only`. The following commands compare the two covariance sources for the non-iterative dual-layer receiver with schedule `[12]`.

Use `duidd_6_6_lmmse_{umi,hls}.cfg`, `duidd_4_4_4_lmmse_{umi,hls}.cfg`, or `duidd_3_3_3_3_lmmse_{umi,hls}.cfg` to repeat the comparison with additional MMSE-PIC stages. Use the `_1ull` `[12]` configurations with `--db 9` and `idd_1ULL.pkl` for the single-layer comparison.

The LMMSE channel-estimation experiments use smaller test sets because of their runtime. Their absolute dataset BLER values should therefore not directly be compared with the other experiments.


In [8]:
for lmmse_config in lmmse_configs:
    command = f'''        
cd {notebook.display_path(REPO / 'scripts')}
python eval_duidd_from_datalake.py --config-name {lmmse_config} --idd-only --db 8 --limit 100 --timestamps idd.pkl
    '''.strip()
    notebook.show_terminal(command)


```bash
$ cd /rx_training/nrx_duidd/scripts
$ python eval_duidd_from_datalake.py --config-name duidd_12_lmmse_idd_umi.cfg --idd-only --db 8 --limit 100 --timestamps idd.pkl
```

```bash
$ cd /rx_training/nrx_duidd/scripts
$ python eval_duidd_from_datalake.py --config-name duidd_12_lmmse_idd_hls.cfg --idd-only --db 8 --limit 100 --timestamps idd.pkl
```

<a id="results"></a>

## 5. Examine the Results

Site-specific parameter adaptation has a larger effect in these experiments. For the non-iterative receiver, replacing synthetic UMi covariance estimates with site-specific estimates reduces absolute dataset BLER by 0.72 for single-layer transmission and 0.26 for dual-layer transmission. With the site-specific LMMSE channel estimator, increasing the number of MMSE-PIC stages reduces dataset BLER from 0.09 at `I=1` to 0.06 at `I=4`, the lowest value reported in the study [[2]](#ref-2).

<p align="left">
  <img src="../../fig/duidd/idd_duidd_lmmse-cov_lab_double_layer-1.png" alt="Site-specific spatial, temporal, and frequency covariance-matrix heatmaps" width="480">
</p>


<a id="references"></a>

## References

<a id="ref-1"></a>[1] R. Wiesmayr, C. Dick, J. Hoydis, and C. Studer, “DUIDD: Deep-Unfolded Interleaved Detection and Decoding for MIMO Wireless Systems,” in *Proc. Asilomar Conference on Signals, Systems, and Computers*, 2022. Available: https://arxiv.org/abs/2212.07816

<a id="ref-2"></a>[2] R. Wiesmayr, N. B. Baytekin, C. Dick, and C. Studer, “On the Impact of Site-Specific Training for a Real-World 5G NR System,” in *Proc. Asilomar Conference on Signals, Systems, and Computers*, 2026.

<a id="ref-3"></a>[3] NVIDIA Corporation, “Aerial CUDA-Accelerated RAN,” release 25-2. Available: https://docs.nvidia.com/aerial/cuda-accelerated-ran/25-2/index.html
